In [1]:
import random
import pickle
from vespa.deployment import VespaCloud
import json
import unicodedata
from dataclasses import dataclass
from typing import Callable, Optional, Iterable, Dict
from vespa.application import Vespa
from time import time
from tqdm.auto import tqdm
import nest_asyncio
from vespa.evaluation import VespaEvaluator
from vespa.package import (
    ApplicationPackage,
    Field,
    Schema,
    Document,
    HNSW,
    RankProfile,
    Component,
    Parameter,
    FieldSet,
    GlobalPhaseRanking,
    Function,
    OnnxModel,
    SecondPhaseRanking,
)

c:\Users\lunar\OneDrive\Área de Trabalho\FGV\7º Período\Séries Temporais\Projeto em CD\MSMARCO\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import requests
import os

# Check if model file already exists
if os.path.exists("model/model.onnx"):
    print("Model file already exists. Skipping download.")
else:
    # Download the ONNX model file
    url = "https://huggingface.co/Xenova/ms-marco-MiniLM-L-6-v2/resolve/main/onnx/model.onnx"
    output_path = "model/model.onnx"

    # Create the directory if it doesn't exist
    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    # Download the file with following redirects
    response = requests.get(url, allow_redirects=True)

    # Save the file
    with open(output_path, 'wb') as f:
        f.write(response.content)

    print(f"Model saved to {output_path}")

Model file already exists. Skipping download.


In [3]:
application = "reranking2"
tenant_name = "segundinho"

In [4]:
class MySchema(Schema):
    @property
    def schema_to_text(self):
        og_text = super().schema_to_text
        # Conserta o caminho do modelo ONNX para usar barras normais
        og_text = og_text.replace("\\", "/")
        return og_text

schema = MySchema(
    name="doc",
    document=Document(
        fields=[
            Field(
                name="id",
                type="string",
                indexing=[
                    "summary",
                    "attribute",
                ],
            ),
            Field(
                name="text",
                type="string",
                indexing=[
                    "summary",
                    "index",
                ],
                index="enable-bm25",
                bolding=True,
            ),
            Field(
                name="text_token_ids",
                type="tensor<float>(d0[64])",
                indexing=[
                    "input text",
                    "embed tokenizer",
                    "attribute",
                ],
                attribute=["paged"],
                is_document_field=False,
            ),
            Field(
                name="embedding",
                type="tensor<float>(x[384])",
                indexing=[
                    "input text",
                    "embed e5",
                    "attribute",
                    "index",
                ],
                ann=HNSW(
                    distance_metric="angular",
                    max_links_per_node=32,
                    neighbors_to_explore_at_insert=400,
                ),
                is_document_field=False,
            ),
            Field(
                name="colbert",
                type="tensor<int8>(dt{}, x[16])",
                indexing=[
                    "input text",
                    "embed colbert",
                    "attribute",
                ],
                attribute=["paged"],
                is_document_field=False,
            ),
        ]
    ),
    fieldsets=[FieldSet(name="default", fields=["text"])],
    rank_profiles=[
        RankProfile(
            name="bm25",
            inputs=[
                ("query(q)", "tensor<float>(x[384])"),
            ],
            first_phase="bm25(text)",
        ),
        RankProfile(
            name="closeness",
            inputs=[
                ("query(q)", "tensor<float>(x[384])"),
            ],
            first_phase="closeness(field, embedding)",
        ),
        RankProfile(
            name="dot-product",
            inputs=[
                ("query(q)", "tensor<float>(x[384])"),
            ],
            functions=[
                Function(name="dot_product", expression="sum(query(q) * attribute(embedding))"),
            ],
            first_phase="dot_product",
        ),
        RankProfile(
            name="fusion-dot-product",
            inherits="dot-product",
            functions=[
                Function(name="fusion_dot_product", expression="0.5*bm25(text) + 0.5*dot_product"),
            ],
            first_phase="fusion_dot_product",
        ),
        RankProfile(
            name="fusion-closeness",
            inherits="closeness",
            functions=[
                Function(name="fusion_closeness", expression="0.5*bm25(text) + 0.5*closeness(field, embedding)"),
            ],
            first_phase="fusion_closeness",
        ),
        RankProfile(
            name="fusion-dot-product-second-phase",
            inherits="fusion-dot-product",
            first_phase="dot_product",
            second_phase=SecondPhaseRanking(
                expression="fusion_dot_product",
                rerank_count=1000,
            ),
        ),
        RankProfile(
            name="fusion-closeness-second-phase",
            inherits="fusion-closeness",
            first_phase="closeness(field, embedding)",
            second_phase=SecondPhaseRanking(
                expression="fusion_closeness",
                rerank_count=1000,
            ),
        ),
        RankProfile(
            name="bm25-colbert",
            inherits="fusion-dot-product-colbert",
            first_phase="bm25(text)",
        ),
        RankProfile(
            name="closeness-colbert",
            inherits="fusion-dot-product-colbert",
            first_phase="closeness(field, embedding)",
        ),
        RankProfile(
            name="dot-product-colbert",
            inherits="fusion-dot-product-colbert",
            first_phase="dot_product",
        ),
        RankProfile(
            name="fusion-dot-product-colbert",
            inherits="fusion-dot-product",
            inputs=[
                ("query(qt)", "tensor<float>(qt{},x[128])"),
                ("query(q)", "tensor<float>(x[384])"),
            ],
            functions=[
                Function(name="cos_sim", expression="cos(distance(field, embedding))"),
                Function(name="max_sim", expression="sum(reduce(sum(query(qt) * unpack_bits(attribute(colbert)), x), max, dt),qt)"),
            ],
            first_phase="fusion_dot_product",
            second_phase=SecondPhaseRanking(
                expression="max_sim",
                rerank_count=100,
            ),
            match_features=["max_sim", "cos_sim"],
        ),
        RankProfile(
            name="fusion-closeness-colbert",
            inherits="fusion-dot-product-colbert",
            functions=[
                Function(name="fusion_closeness", expression="0.5*bm25(text) + 0.5*closeness(field, embedding)"),
            ],
            first_phase="fusion_closeness",
        ),
        RankProfile(
            name="bm25-cross-encoder",
            inherits="fusion-dot-product-cross-encoder",
            first_phase="bm25(text)",
        ),
        RankProfile(
            name="closeness-cross-encoder",
            inherits="fusion-dot-product-cross-encoder",
            first_phase="closeness(field, embedding)",
        ),
        RankProfile(
            name="dot-product-cross-encoder",
            inherits="fusion-dot-product-cross-encoder",
            first_phase="dot_product",
        ),
        RankProfile(
            name="fusion-dot-product-cross-encoder",
            inherits="fusion-dot-product",
            inputs=[
                ("query(q)", "tensor<float>(x[384])"),
                ("query(query_token_ids)",  "tensor<float>(d0[32])"),
            ],
            functions=[
                Function(name="input_ids", expression="tokenInputIds(96, query(query_token_ids), attribute(text_token_ids))"),
                Function(name="token_type_ids", expression="tokenTypeIds(96, query(query_token_ids), attribute(text_token_ids))"),
                Function(name="attention_mask", expression="tokenAttentionMask(96, query(query_token_ids), attribute(text_token_ids))"),
                Function(name="cross_encoder", expression="onnx(cross_encoder_model){d0:0,d1:0}")
            ],
            first_phase="fusion_dot_product",
            second_phase=SecondPhaseRanking(
                expression="cross_encoder",
                rerank_count=100
            )
        ),
        RankProfile(
            name="fusion-closeness-cross-encoder",
            inherits="fusion-dot-product-cross-encoder",
            functions=[
                Function(name="fusion_closeness", expression="0.5*bm25(text) + 0.5*closeness(field, embedding)"),
            ],
            first_phase="fusion_closeness",
        ),
    ],
    models=[
        OnnxModel(
            model_name="cross_encoder_model",
            model_file_path="model/model.onnx",
            inputs={
                "input_ids": "input_ids",
                "attention_mask": "attention_mask",
                "token_type_ids": "token_type_ids",
            },
            outputs={"logits": "logits"},
        ),
    ],
)

package = ApplicationPackage(
    name=application,
    schema=[
        schema
    ],
    components=[
        Component(
            id="e5",
            type="hugging-face-embedder",
            parameters=[
                Parameter(
                    "transformer-model",
                    {
                        "url": "https://huggingface.co/intfloat/e5-small-v2/resolve/main/model.onnx"
                    },
                ),
                Parameter(
                    "tokenizer-model",
                    {
                        "url": "https://huggingface.co/intfloat/e5-small-v2/raw/main/tokenizer.json"
                    },
                ),
            ],
        ),
        Component(
            id="colbert",
            type="colbert-embedder",
            parameters=[
                Parameter(
                    "transformer-model",
                    {
                        "url": "https://huggingface.co/colbert-ir/colbertv2.0/resolve/main/model.onnx"
                    },
                ),
                Parameter(
                    "tokenizer-model",
                    {
                        "url": "https://huggingface.co/colbert-ir/colbertv2.0/raw/main/tokenizer.json"
                    },
                ),
            ],
        ),
        Component(
            id="tokenizer",
            type="hugging-face-tokenizer",
            parameters=[
                Parameter(
                    "model",
                    {
                        "url": "https://huggingface.co/Xenova/ms-marco-MiniLM-L-6-v2/raw/main/tokenizer.json"
                    },
                ),
            ],
        ),
    ],
)

In [5]:
from vespa.deployment import VespaDocker
from vespa.application import Vespa

vespa_cloud = VespaCloud(
    tenant=tenant_name,
    application=application,
    application_package=package,
)
# app = vespa_cloud.deploy()
app = vespa_cloud.get_application()

Setting application...
Running: vespa config set application segundinho.reranking2.default
Setting target cloud...
Running: vespa config set target cloud

No api-key found for control plane access. Using access token.
Checking for access token in auth.json...
Successfully obtained access token for control plane access.
Only region: aws-us-east-1c available in dev environment.
Found mtls endpoint for reranking2_container
URL: https://e1d2987c.aeb4d96e.z.vespa-app.cloud/
Application is up!


In [6]:
# Splita as queries de treinamento e teste

random.seed(42)

def load_dataset(input_file):
    with open(input_file, 'rb') as f:
        return pickle.load(f)

data_set = "../subset_msmarco_train_0/subset_msmarco_train_0.01_99.pkl"

data = load_dataset(data_set)
queries = data["queries"]
documents = data["docs"]
qrels = data["qrels"]

# Split the queries (queries is a dictionary of {query_id: query_object})
query_ids = list(queries.keys())  # List of query IDs

# Shuffle query IDs to ensure a random split
random.shuffle(query_ids)

# Split into 80% for training, 20% for validation
split_ratio = 0.8
train_query_ids = query_ids[:int(len(query_ids) * split_ratio)]
test_query_ids = query_ids[int(len(query_ids) * split_ratio):]

train_queries = {qid: queries[qid] for qid in train_query_ids}
test_queries = {qid: queries[qid] for qid in test_query_ids}

In [7]:
# Define feed parameters for the Vespa application
@dataclass
class FeedParams:
    name: str
    num_docs: int
    max_connections: int
    function_name: str
    max_workers: Optional[int] = None
    max_queue_size: Optional[int] = None


@dataclass
class FeedResult(FeedParams):
    feed_time: Optional[float] = None

In [8]:
import re
def remove_control_characters(text):
    # Remove control characters using regex
    return re.sub(r'[\x00-\x1F\x7F-\x9F]', '', text)

In [9]:
namespace = "default"
doctype = "doc"

vespa_docs = []

for doc_id, doc_obj in documents.items():
    vespa_doc = {
        "put": f"id:{namespace}:{doctype}::{doc_id}",
        "fields": {
            "id": str(doc_id),
            "text": remove_control_characters(doc_obj.text),
        }
    }
    vespa_docs.append(vespa_doc)

feed_file = "vespa_feed.json"

with open(feed_file, "w", encoding="utf-8") as f:
    json.dump(vespa_docs, f, ensure_ascii=False)

print(f"✅ {len(vespa_docs)} documentos salvos em {feed_file}")

✅ 277168 documentos salvos em vespa_feed.json


In [10]:
# output_list = !vespa feed vespa_feed.json
# results = json.loads("".join(output_list))
# print(results)

In [29]:
import types
from typing import Dict, Iterable
from vespa.application import Vespa
import httpx

def new_query_many(
        self,
        queries: Iterable[Dict],
        num_connections: int = 1,
        max_concurrent: int = 4,
        client_kwargs: Dict = {},
        **query_kwargs,
    ):
    client_kwargs = dict(client_kwargs)  # copy to avoid mutating input
    client_kwargs["timeout"] = httpx.Timeout(600.0)  # 10 minutes
    return Vespa.query_many(
        self,
        queries=queries,
        num_connections=num_connections,
        max_concurrent=max_concurrent,
        client_kwargs=client_kwargs,
        **query_kwargs,
    )

# Patch the Vespa app instance to use the new query_many method
app.query_many = types.MethodType(new_query_many, app)

In [31]:
test_queries_dict = {
    q.query_id: q.text
    for q in test_queries.values()
}

relevant_docs = dict()
for qrel in qrels:
    relevant_docs[qrel.query_id] = relevant_docs.get(qrel.query_id, set())
    relevant_docs[qrel.query_id].add(qrel.doc_id)

In [ ]:

    
def create_colbert_query_fn(top_k=10, timeout=600):
    def query_fn(query_text: str, k: int = top_k) -> dict:
        return {
            "yql": "select * from sources * where userQuery() or ({targetHits:1000}nearestNeighbor(embedding,q))",
            "query": query_text,
            "ranking": "fusion-dot-product-colbert",  # corrected to use the proper colbert profile
            "ranking.features.query(q)": f"embed(e5, '{query_text}')",
            "ranking.features.query(qt)": f"embed(colbert, '{query_text}')",  # required for ColBERT
            "ranking.features.query(query_token_ids)": f"embed(tokenizer, '{query_text}')",
            "timeout": timeout,
            "ranking.softtimeout.enable": "false",
            "hits": k,
        }
    return query_fn

vespa_query_fn = create_colbert_query_fn(top_k=10)

evaluator = VespaEvaluator(
    queries=test_queries_dict,
    relevant_docs=relevant_docs,
    vespa_query_fn=vespa_query_fn,
    app=app,
    name="test-run-colbert-reranking",
    accuracy_at_k=[10],
    precision_recall_at_k=[10],
    mrr_at_k=[10],
    ndcg_at_k=[10],
    write_csv=True,
)

results = evaluator.run()

print("Results for colbert reranking:")
print("Primary Metric:", evaluator.primary_metric)
print("Results:", results)

Results for colbert reranking:
Primary Metric: ndcg@10
Results: {'accuracy@10': 0.9063063063063063, 'precision@10': 0.09423423423423506, 'recall@10': 0.9045045045045045, 'mrr@10': 0.7595359645359644, 'ndcg@10': 0.7929975012412607, 'map@100': 0.7560403648648018, 'searchtime_avg': 0.09529909909909906, 'searchtime_q50': 0.1, 'searchtime_q90': 0.14360000000000003, 'searchtime_q95': 0.15429999999999996}


In [30]:
def create_fusion_dot_product_query_fn(top_k=10, timeout=600):
    def query_fn(query_text: str, k: int = top_k) -> dict:
        return {
            "yql": "select * from sources * where userQuery() or ({targetHits:1000}nearestNeighbor(embedding,q))",
            "query": query_text,
            "ranking": "fusion-dot-product-cross-encoder",
            "ranking.features.query(q)": f"embed(e5, '{query_text}')",
            "ranking.features.query(query_token_ids)": f"embed(tokenizer, '{query_text}')",
            "timeout": f"{timeout}s",
            "ranking.softtimeout.enable": "false",
            "hits": k,
        }
    return query_fn

vespa_query_fn = create_fusion_dot_product_query_fn(top_k=10)

evaluator = VespaEvaluator(
    queries=test_queries_dict,
    relevant_docs=relevant_docs,
    vespa_query_fn=vespa_query_fn,
    app=app,
    name="test-run-fusion-dot-product-cross-encoder",
    accuracy_at_k=[10],
    precision_recall_at_k=[10],
    mrr_at_k=[10],
    ndcg_at_k=[10],
    write_csv=True,
)

results = evaluator.run()

print("Results for fusion dot product cross-encoder reranking:")
print("Primary Metric:", evaluator.primary_metric)
print("Results:", results)

Results for fusion dot product cross-encoder reranking:
Primary Metric: ndcg@10
Results: {'accuracy@10': 0.9009009009009009, 'precision@10': 0.09351351351351433, 'recall@10': 0.8981981981981982, 'mrr@10': 0.7677634777634776, 'ndcg@10': 0.7977723251725579, 'map@100': 0.7654635404635403, 'searchtime_avg': 5.170841441441437, 'searchtime_q50': 5.1850000000000005, 'searchtime_q90': 6.194800000000002, 'searchtime_q95': 7.696199999999999}


In [32]:
def create_closeness_colbert_query_fn(top_k=10, timeout=600):
    def query_fn(query_text: str, k: int = top_k) -> dict:
        return {
            "yql": "select * from sources * where userQuery() or ({targetHits:1000}nearestNeighbor(embedding,q))",
            "query": query_text,
            "ranking": "closeness-colbert",
            "ranking.features.query(q)": f"embed(e5, '{query_text}')",
            "ranking.features.query(qt)": f"embed(colbert, '{query_text}')",
            "timeout": timeout,
            "ranking.softtimeout.enable": "false",
            "hits": k,
        }
    return query_fn

vespa_query_fn = create_closeness_colbert_query_fn(top_k=10)

evaluator = VespaEvaluator(
    queries=test_queries_dict,
    relevant_docs=relevant_docs,
    vespa_query_fn=vespa_query_fn,
    app=app,
    name="test-run-closeness-colbert",
    accuracy_at_k=[10],
    precision_recall_at_k=[10],
    mrr_at_k=[10],
    ndcg_at_k=[10],
    write_csv=True,
)

results = evaluator.run()

print("Results for colbert with semantic closeness:")
print("Primary Metric:", evaluator.primary_metric)
print("Results:", results)

Results for colbert with semantic closeness:
Primary Metric: ndcg@10
Results: {'accuracy@10': 0.9135135135135135, 'precision@10': 0.09513513513513597, 'recall@10': 0.9126126126126126, 'mrr@10': 0.7476383526383527, 'ndcg@10': 0.7858639930526439, 'map@100': 0.7448142925591781, 'searchtime_avg': 0.08153873873873888, 'searchtime_q50': 0.082, 'searchtime_q90': 0.10200000000000001, 'searchtime_q95': 0.107}


In [33]:
def create_closeness_cross_encoder_query_fn(top_k=10, timeout=600):
    def query_fn(query_text: str, k: int = top_k) -> dict:
        return {
            "yql": "select * from sources * where userQuery() or ({targetHits:1000}nearestNeighbor(embedding,q))",
            "query": query_text,
            "ranking": "closeness-cross-encoder",
            "ranking.features.query(q)": f"embed(e5, '{query_text}')",
            "ranking.features.query(query_token_ids)": f"embed(tokenizer, '{query_text}')",
            "timeout": f"{timeout}s",
            "ranking.softtimeout.enable": "false",
            "hits": k,
        }
    return query_fn

vespa_query_fn = create_closeness_cross_encoder_query_fn(top_k=10)

evaluator = VespaEvaluator(
    queries=test_queries_dict,
    relevant_docs=relevant_docs,
    vespa_query_fn=vespa_query_fn,
    app=app,
    name="test-run-closeness-cross-encoder",
    accuracy_at_k=[10],
    precision_recall_at_k=[10],
    mrr_at_k=[10],
    ndcg_at_k=[10],
    write_csv=True,
)

results = evaluator.run()

print("Results for cross encoder with semantic closeness:")
print("Primary Metric:", evaluator.primary_metric)
print("Results:", results)

Results for cross encoder with semantic closeness:
Primary Metric: ndcg@10
Results: {'accuracy@10': 0.9081081081081082, 'precision@10': 0.09441441441441524, 'recall@10': 0.9063063063063063, 'mrr@10': 0.7631660231660228, 'ndcg@10': 0.7966352067020058, 'map@100': 0.7619155229819278, 'searchtime_avg': 5.2447981981982, 'searchtime_q50': 5.319, 'searchtime_q90': 6.448800000000002, 'searchtime_q95': 7.891299999999999}
